# Actividad de Estadística Inferencial en R con datos MEPS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/Astroestadistica/blob/main/ITM/Actividad_ITM_MEPS_R.ipynb)

**Tema elegido:** Acceso a seguro de salud y condiciones sociodemográficas en la Medical Expenditure Panel Survey (MEPS).  
**Fuente del dataset:** `AER::HealthInsurance` (MEPS) vía CSV público en Rdatasets.

---

## Introducción

En esta actividad se desarrolla un análisis estadístico inferencial **completo y extenso** en **R** usando datos de la **Medical Expenditure Panel Survey (MEPS)**. El propósito es estudiar cómo características demográficas y socioeconómicas se asocian con la probabilidad de contar con seguro de salud.

El cuaderno incluye:

1. Carga y limpieza de datos.
2. Muestreo estratificado de 70 observaciones (siguiendo el enfoque de la guía ITM).
3. Análisis descriptivo numérico y gráfico.
4. Pruebas de hipótesis (normalidad, independencia, comparación de grupos, proporciones).
5. Modelo de regresión logística para inferencia y predicción.
6. Conclusiones aplicadas.


In [ ]:
# Configuración inicial
set.seed(2026)

required_pkgs <- c("dplyr", "ggplot2", "tidyr", "forcats", "broom", "scales")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs, repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(forcats)
  library(broom)
  library(scales)
})


## 1) Carga del dataset MEPS y revisión inicial

Se usa el dataset **HealthInsurance**, documentado como datos provenientes de MEPS. Este conjunto permite analizar variables cualitativas clave (estado de salud, seguro, estado civil, región, etnia, educación) y variables cuantitativas (`age`, `family`).


In [ ]:
url <- "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/HealthInsurance.csv"
meps <- read.csv(url)

# Limpieza básica
df <- meps %>%
  select(-rownames) %>%
  mutate(
    across(c(health, limit, gender, insurance, married, selfemp, region, ethnicity, education), as.factor),
    age = as.numeric(age),
    family = as.numeric(family)
  )

cat("Observaciones totales:", nrow(df), "
")
cat("Variables:", ncol(df), "

")
str(df)
summary(df)


## 2) Selección de 70 observaciones sin sesgo (muestreo estratificado)

Para mantener representatividad del conjunto original, se estratifica por la combinación de:

- `insurance` (sí/no)
- `region` (4 regiones)

Luego se ajusta exactamente a 70 casos distribuyendo los decimales de forma controlada.


In [ ]:
strata <- df %>%
  count(insurance, region, name = "N") %>%
  mutate(
    exact = N / sum(N) * 70,
    n_take = floor(exact),
    frac = exact - n_take
  )

resto <- 70 - sum(strata$n_take)
if (resto > 0) {
  strata <- strata %>%
    arrange(desc(frac)) %>%
    mutate(n_take = n_take + ifelse(row_number() <= resto, 1, 0))
}

sample_70 <- strata %>%
  select(insurance, region, n_take) %>%
  left_join(df, by = c("insurance", "region")) %>%
  group_by(insurance, region) %>%
  slice_sample(n = first(n_take)) %>%
  ungroup()

cat("Tamaño muestra final:", nrow(sample_70), "
")
sample_70 %>% count(insurance, region)


### Verificación de representatividad de la muestra

El gráfico compara proporciones por estrato entre el conjunto completo y la muestra de 70 observaciones.


In [ ]:
comp <- bind_rows(
  df %>% count(insurance, region) %>% mutate(source = "Completo"),
  sample_70 %>% count(insurance, region) %>% mutate(source = "Muestra 70")
) %>%
  group_by(source) %>%
  mutate(prop = n / sum(n)) %>%
  ungroup() %>%
  mutate(estrato = paste(insurance, region, sep = " | "))

p_rep <- ggplot(comp, aes(x = fct_reorder(estrato, prop), y = prop, fill = source)) +
  geom_col(position = "dodge") +
  coord_flip() +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(
    title = "Comparación de composición por estrato",
    x = "Estrato (insurance | region)",
    y = "Proporción"
  ) +
  theme_minimal(base_size = 13)

p_rep


## 3) Definición de variables para la actividad

En la muestra de 70 se cumplen los requisitos de variedad de variables:

- **Cuantitativas continuas/discretas:** `age`, `family`.
- **Cualitativas nominales:** `gender`, `insurance`, `ethnicity`, `region`, `selfemp`, `married`, `limit`.
- **Cualitativa ordinal:** `education`.

Se construye además una versión ordenada de educación para análisis no paramétricos y visualización.


In [ ]:
edu_levels <- c("<highschool", "highschool", "college", ">college")

sample_70 <- sample_70 %>%
  mutate(
    education_ord = factor(education, levels = edu_levels, ordered = TRUE),
    insured_bin = ifelse(insurance == "yes", 1, 0)
  )

summary(sample_70 %>% select(age, family, insured_bin, education_ord))


## 4) Análisis descriptivo (numérico + gráfico)

Se reportan estadísticas de tendencia central/dispersión y múltiples gráficos para facilitar la comprensión de patrones.


In [ ]:
desc_num <- sample_70 %>%
  summarise(
    n = n(),
    age_media = mean(age),
    age_mediana = median(age),
    age_sd = sd(age),
    age_min = min(age),
    age_max = max(age),
    family_media = mean(family),
    family_mediana = median(family),
    family_sd = sd(family),
    family_min = min(family),
    family_max = max(family)
  )

desc_num


In [ ]:
# Gráfico 1: distribución de edad
p1 <- ggplot(sample_70, aes(x = age)) +
  geom_histogram(binwidth = 5, fill = "#4472C4", color = "white", alpha = 0.9) +
  geom_vline(aes(xintercept = mean(age)), linetype = "dashed", color = "#D62728", linewidth = 1) +
  labs(title = "Distribución de edad en la muestra", x = "Edad", y = "Frecuencia") +
  theme_minimal(base_size = 13)

# Gráfico 2: distribución de tamaño familiar
p2 <- ggplot(sample_70, aes(x = family)) +
  geom_histogram(binwidth = 1, fill = "#2CA02C", color = "white", alpha = 0.9) +
  labs(title = "Distribución de tamaño del hogar (family)", x = "Número de personas", y = "Frecuencia") +
  theme_minimal(base_size = 13)

# Gráfico 3: edad por cobertura de seguro
p3 <- ggplot(sample_70, aes(x = insurance, y = age, fill = insurance)) +
  geom_boxplot(alpha = 0.85, outlier.color = "#D62728") +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Edad según condición de seguro", x = "Seguro", y = "Edad") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

# Gráfico 4: proporción de seguro por género
p4 <- sample_70 %>%
  count(gender, insurance) %>%
  group_by(gender) %>%
  mutate(prop = n / sum(n)) %>%
  ggplot(aes(x = gender, y = prop, fill = insurance)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  scale_fill_brewer(palette = "Pastel1") +
  labs(title = "Composición de seguro por género", x = "Género", y = "Proporción") +
  theme_minimal(base_size = 13)

# Gráfico 5: mapa de calor de frecuencia insurance vs limit
p5 <- sample_70 %>%
  count(insurance, limit) %>%
  ggplot(aes(x = insurance, y = limit, fill = n)) +
  geom_tile(color = "white") +
  geom_text(aes(label = n), color = "black", size = 5) +
  scale_fill_gradient(low = "#EFF3FF", high = "#08519C") +
  labs(title = "Frecuencias conjuntas: seguro y limitaciones", x = "Seguro", y = "Limit") +
  theme_minimal(base_size = 13)

p1
p2
p3
p4
p5


### Hallazgos descriptivos clave

1. La edad presenta dispersión amplia, lo que permite comparar subgrupos con diferente ciclo de vida.
2. El tamaño familiar (`family`) se concentra en valores bajos-medios, con cola hacia hogares más grandes.
3. Se observan diferencias visuales entre grupos con y sin seguro.
4. La visualización por género sugiere variaciones en composición de cobertura.
5. La tabla de calor `insurance` vs `limit` ayuda a identificar rápidamente concentraciones y posibles asociaciones.


## 5) Pruebas de hipótesis

Se usa \(lpha = 0.05\) para todas las decisiones inferenciales.


In [ ]:
alpha <- 0.05

# 1) Normalidad para edad (Shapiro-Wilk)
shapiro_age <- shapiro.test(sample_70$age)

# 2) Independencia entre insurance y limit (Chi-cuadrado)
tbl_ins_limit <- table(sample_70$insurance, sample_70$limit)
chi_ins_limit <- chisq.test(tbl_ins_limit)

# 3) Comparación de edad por insurance (Mann-Whitney)
wilcox_age_ins <- wilcox.test(age ~ insurance, data = sample_70)

# 4) Diferencia de edad por nivel educativo (Kruskal-Wallis)
kruskal_age_edu <- kruskal.test(age ~ education_ord, data = sample_70)

# 5) Diferencia de proporciones de seguro entre géneros
prop_gender <- sample_70 %>%
  group_by(gender) %>%
  summarise(insured = sum(insurance == "yes"), n = n(), .groups = "drop")
prop_gender_test <- prop.test(x = prop_gender$insured, n = prop_gender$n)

# Resumen de resultados
infer_summary <- tibble::tibble(
  prueba = c(
    "Shapiro-Wilk (age)",
    "Chi-cuadrado (insurance vs limit)",
    "Mann-Whitney (age ~ insurance)",
    "Kruskal-Wallis (age ~ education)",
    "Prop.test (insurance por gender)"
  ),
  p_value = c(
    shapiro_age$p.value,
    chi_ins_limit$p.value,
    wilcox_age_ins$p.value,
    kruskal_age_edu$p.value,
    prop_gender_test$p.value
  )
) %>%
  mutate(decision_5pct = ifelse(p_value < alpha, "Rechazar H0", "No rechazar H0"))

infer_summary


### Interpretación inferencial

- **Shapiro-Wilk** evalúa si `age` es compatible con normalidad.
- **Chi-cuadrado** evalúa asociación entre tenencia de seguro y estado de `limit`.
- **Mann-Whitney** contrasta diferencias de edad entre quienes tienen/no tienen seguro sin asumir normalidad estricta.
- **Kruskal-Wallis** evalúa si la distribución de edad difiere por nivel educativo.
- **Prueba de proporciones** contrasta la fracción de personas aseguradas entre géneros.

La decisión se basa en el p-valor frente a \(lpha = 0.05\).


## 6) Asociación entre variables cuantitativas y modelado inferencial

Aunque hay pocas variables numéricas, se evalúa relación monotónica `age`-`family` y luego se ajusta un modelo de **regresión logística** para la probabilidad de tener seguro.


In [ ]:
# Correlación de Spearman entre age y family
cor_age_family <- cor.test(sample_70$age, sample_70$family, method = "spearman")
cor_age_family


In [ ]:
# Regresión logística para explicar insurance
model_logit <- glm(
  insurance ~ age + gender + married + selfemp + family + region + ethnicity + education_ord + health + limit,
  data = sample_70,
  family = binomial(link = "logit")
)

summary(model_logit)

# Pseudo R2 de McFadden
model_null <- glm(insurance ~ 1, data = sample_70, family = binomial(link = "logit"))
pseudo_r2 <- 1 - as.numeric(logLik(model_logit) / logLik(model_null))
pseudo_r2


In [ ]:
# Tabla de odds ratios con IC 95%
or_tbl <- broom::tidy(model_logit, conf.int = TRUE, exponentiate = TRUE) %>%
  filter(term != "(Intercept)") %>%
  mutate(term = gsub("education_ord", "education:", term))

or_tbl

# Gráfico de odds ratios
p_or <- ggplot(or_tbl, aes(x = reorder(term, estimate), y = estimate)) +
  geom_point(color = "#1F77B4", size = 2.8) +
  geom_errorbar(aes(ymin = conf.low, ymax = conf.high), width = 0.12, color = "#1F77B4") +
  geom_hline(yintercept = 1, linetype = "dashed", color = "#D62728") +
  coord_flip() +
  scale_y_log10() +
  labs(
    title = "Odds Ratios del modelo logístico (escala log)",
    x = "Predictor",
    y = "Odds Ratio"
  ) +
  theme_minimal(base_size = 12)

p_or


## 7) Predicción aplicada con perfiles hipotéticos

Se generan perfiles para ilustrar el uso del modelo en decisiones de política y gestión.


In [ ]:
new_profiles <- data.frame(
  age = c(25, 40, 58, 35),
  gender = factor(c("female", "male", "female", "male"), levels = levels(sample_70$gender)),
  married = factor(c("no", "yes", "yes", "no"), levels = levels(sample_70$married)),
  selfemp = factor(c("no", "no", "yes", "yes"), levels = levels(sample_70$selfemp)),
  family = c(1, 3, 2, 4),
  region = factor(c("northeast", "south", "midwest", "west"), levels = levels(sample_70$region)),
  ethnicity = factor(c("white", "white", "black", "hispanic"), levels = levels(sample_70$ethnicity)),
  education_ord = factor(c("college", "highschool", ">college", "<highschool"), levels = levels(sample_70$education_ord), ordered = TRUE),
  health = factor(c("yes", "yes", "no", "yes"), levels = levels(sample_70$health)),
  limit = factor(c("no", "no", "yes", "yes"), levels = levels(sample_70$limit))
)

pred <- predict(model_logit, newdata = new_profiles, type = "response")
pred_df <- cbind(new_profiles, prob_insurance = pred)
pred_df


In [ ]:
p_pred <- pred_df %>%
  mutate(perfil = paste0("Perfil ", row_number())) %>%
  ggplot(aes(x = perfil, y = prob_insurance, fill = perfil)) +
  geom_col(width = 0.65, alpha = 0.9) +
  geom_text(aes(label = percent(prob_insurance, accuracy = 0.1)), vjust = -0.3, size = 4) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1.05)) +
  labs(title = "Probabilidad estimada de tener seguro por perfil", x = "Perfiles", y = "Probabilidad") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

p_pred


## 8) Conclusiones

1. El muestreo estratificado permitió construir una muestra de 70 observaciones con estructura cercana al dataset completo de MEPS en variables críticas (`insurance`, `region`).
2. El análisis descriptivo y gráfico mostró heterogeneidad en edad, tamaño familiar y distribución de cobertura de seguro por subgrupos.
3. Las pruebas de hipótesis permiten identificar qué asociaciones/diferencias son estadísticamente sostenibles y cuáles no, bajo \(lpha = 0.05\).
4. La regresión logística cuantifica efectos parciales sobre la probabilidad de tener seguro, aportando una base sólida para inferencia y toma de decisiones.
5. El enfoque combinado (descriptivo + inferencial + predictivo) facilita una lectura integral de los hallazgos en contexto MEPS.


## 9) Referencias de datos

- Kleiber, C., & Zeileis, A. (2008). *Applied Econometrics with R*. Springer.  
- Dataset `HealthInsurance` (MEPS) disponible en Rdatasets:  
  `https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/HealthInsurance.csv`
